In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("PartitionDemo").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 04:45:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/13 04:45:17 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
data = [(i, i*10) for i in range(1, 21)]

In [4]:
df = spark.createDataFrame(data, ["id", "value"])


In [5]:
print("Before Partitions:", df.rdd.getNumPartitions())

Before Partitions: 2


In [8]:
df_repartitioned = df.repartition(20)

In [9]:
print("After Repartition:", df_repartitioned.rdd.getNumPartitions())

After Repartition: 20


In [10]:
df_coalesced = df_repartitioned.coalesce(2)

In [11]:
print("After Coalesced:", df_coalesced.rdd.getNumPartitions())

After Coalesced: 2


In [12]:
with open("students.csv", "w") as f:
    f.write("""Id,Name,Course,Marks,Age
1,Akshat,Python,85,21
2,Rahul,Java,78,22
3,Priya,Data Science,92,20
4,Neha,Python,88,23
5,Amit,Java,75,21
6,Simran,Data Science,95,22
7,Rohit,Python,81,20
8,Karan,Java,89,24
9,Pooja,Data Science,91,23
10,Ankit,Python,77,22
""")

In [13]:
df = spark.read \
    .option("header", True) \
    .csv("students.csv")

df.show()

+---+------+------------+-----+---+
| Id|  Name|      Course|Marks|Age|
+---+------+------------+-----+---+
|  1|Akshat|      Python|   85| 21|
|  2| Rahul|        Java|   78| 22|
|  3| Priya|Data Science|   92| 20|
|  4|  Neha|      Python|   88| 23|
|  5|  Amit|        Java|   75| 21|
|  6|Simran|Data Science|   95| 22|
|  7| Rohit|      Python|   81| 20|
|  8| Karan|        Java|   89| 24|
|  9| Pooja|Data Science|   91| 23|
| 10| Ankit|      Python|   77| 22|
+---+------+------------+-----+---+



In [14]:
df.createOrReplaceTempView("students")

In [15]:
spark.sql("""
SELECT *
FROM students
WHERE Marks > 85
""").show()

+---+------+------------+-----+---+
| Id|  Name|      Course|Marks|Age|
+---+------+------------+-----+---+
|  3| Priya|Data Science|   92| 20|
|  4|  Neha|      Python|   88| 23|
|  6|Simran|Data Science|   95| 22|
|  8| Karan|        Java|   89| 24|
|  9| Pooja|Data Science|   91| 23|
+---+------+------------+-----+---+



In [16]:
df.write.mode("overwrite").csv("student_data")